## HIL-007 Data Inspection and Cleaning

This notebook documents a conservative cleaning workflow for the HIL-007 pavement projects snapshot. The raw ArcGIS response is preserved unchanged; derived cleaned attributes, geometry, and quality flags are exported separately.

- Source: City of Hillsboro GIS
- Dataset: Pavement projects
- Snapshot: `2026-09-08`
- Geometry: Polyline
- Coordinate system: NAD83(HARN) Oregon State Plane North, US Survey Feet

In [8]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
from shapely.geometry import LineString, MultiLineString

# Select the dataset from the notebook name and use the newest acquired snapshot.
NOTEBOOK_DIR = Path.cwd()
DATASET_ID = "HIL-007"
DATASET_FILES = sorted(
    NOTEBOOK_DIR.glob(f"*/datasets/raw/{DATASET_ID}.json"),
    key=lambda path: path.parent.parent.parent.name,
    reverse=True,
)
if not DATASET_FILES:
    raise FileNotFoundError(
        f"No acquired snapshot found for {DATASET_ID} below {NOTEBOOK_DIR}"
    )

RAW_FILE = DATASET_FILES[0]
DATA_VERSION = RAW_FILE.parent.parent.parent.name
MANIFESTS_DIR = RAW_FILE.parent.parent.parent / "manifests"
MANIFEST_FILE = MANIFESTS_DIR / f"{DATASET_ID}_manifest.json"
PROCESSED_DIR = RAW_FILE.parent.parent / "processed"
PROCESSED_FILE = PROCESSED_DIR / f"{DATASET_ID}_cleaned.json"
CLEANING_MANIFEST_FILE = MANIFESTS_DIR / f"{DATASET_ID}_cleaning_manifest.json"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

with open(MANIFEST_FILE, "r", encoding="utf-8") as file:
    source_manifest = json.load(file)

df = pd.DataFrame([feature["attributes"] for feature in raw_data["features"]])

print(f"Loaded {RAW_FILE.name} ({DATA_VERSION}): {len(df):,} rows, {len(df.columns):,} source fields")
print(f"Geometry type: {raw_data.get('geometryType')}")

Loaded HIL-007.json (2026-09-08): 66 rows, 41 source fields
Geometry type: esriGeometryPolyline


## Raw Structure and Missingness

The initial audit checks field types, missingness, identifier uniqueness, pavement condition ranges, and the source geometry structure before transformation.

In [2]:
schema_df = pd.DataFrame(raw_data["fields"])[["name", "alias", "type"]]
missing = df.isna().sum().to_frame("missing_count")
missing["missing_percent"] = (missing["missing_count"] / len(df) * 100).round(1)
identifier_fields = [field for field in ["OBJECTID", "GlobalID"] if field in df]
identifier_audit = pd.DataFrame({
    "field": identifier_fields,
    "unique_values": [df[field].nunique() for field in identifier_fields],
    "missing_values": [df[field].isna().sum() for field in identifier_fields],
    "duplicate_values": [df[field].duplicated().sum() for field in identifier_fields],
})

print("Field types:")
display(schema_df)
print("Missingness:")
display(missing.sort_values("missing_count", ascending=False))
print("Identifier audit:")
display(identifier_audit)
print("Pavement condition index range:", int(df["PAVEMENT_CONDITION_INDEX"].min()), "-", int(df["PAVEMENT_CONDITION_INDEX"].max()))
print("Negative Shape__Length values:", int((df["Shape__Length"] < 0).sum()))

Field types:


,name,alias,type
0,GlobalID,GlobalID,esriFieldTypeGlobalID
1,CREATION_DATE,Creation Date,esriFieldTypeDate
2,CREATOR,Creator,esriFieldTypeString
3,LAST_UPDATE,Last Update,esriFieldTypeDate
4,LATEST_CONSTRUCTION_DATE,Latest Construction Date,esriFieldTypeDate
5,LATEST_MAINTENANCE_DATE,Latest Maintenance Date,esriFieldTypeDate
6,MORATORIUM_EXPIRATION_DATE,Moratorium Expiration Date,esriFieldTypeDate
7,ORIGINAL_CONSTRUCTION_DATE,Original Construction Date,esriFieldTypeDate
8,PAVEMENT_CONDITION_INDEX,Pavement Condition Index,esriFieldTypeSmallInteger
9,PCI_DATE,PCI Date,esriFieldTypeDate


Missingness:


,missing_count,missing_percent
MORATORIUM_EXPIRATION_DATE,66,100.0
CORRIDOR,66,100.0
MORATORIUM_STATUS,66,100.0
STORM_STATUS,66,100.0
PW_ROADWAY_ID,66,100.0
GIS_COMMENTS,66,100.0
DESCRIPTION1,66,100.0
ASBUILT_REFERENCE,66,100.0
CREATOR,54,81.8
SECONDARY_PLANNED_MAINTENANCE,54,81.8


Identifier audit:


,field,unique_values,missing_values,duplicate_values
0,OBJECTID,66,0,0
1,GlobalID,66,0,0


Pavement condition index range: 63 - 94
Negative Shape__Length values: 0


## Geometry and Derived Clean View

Polyline paths are converted for validation only. Invalid geometries are repaired in the derived view with `make_valid`; source paths and source attributes remain unchanged.

In [3]:
def arcgis_polyline_to_shapely(geometry):
    paths = geometry.get("paths", []) if geometry else []
    lines = [LineString(path) for path in paths if len(path) >= 2]
    if not lines:
        return None
    return lines[0] if len(lines) == 1 else MultiLineString(lines)

geometry_records = []
for feature in raw_data["features"]:
    geometry = feature.get("geometry") or {}
    paths = geometry.get("paths", [])
    line = arcgis_polyline_to_shapely(geometry)
    geometry_records.append({
        "geometry": line,
        "path_count": len(paths),
        "vertex_count": sum(len(path) for path in paths),
    })

geometry_df = pd.DataFrame(geometry_records)
geometry_df["valid_source_geometry"] = geometry_df["geometry"].map(
    lambda geometry: geometry is not None and geometry.is_valid
)
geometry_df["empty_geometry_flag"] = geometry_df["geometry"].isna()
geometry_df["length"] = geometry_df["geometry"].map(
    lambda geometry: geometry.length if geometry is not None else np.nan
)

print("Geometry records:", len(geometry_df))
print("Invalid geometries:", int((~geometry_df["valid_source_geometry"] & ~geometry_df["empty_geometry_flag"]).sum()))
print("Empty geometries:", int(geometry_df["empty_geometry_flag"].sum()))
print("Multi-path records:", int((geometry_df["path_count"] > 1).sum()))

Geometry records: 66
Invalid geometries: 0
Empty geometries: 0
Multi-path records: 0


In [4]:
from shapely.validation import make_valid

analysis_df = df.copy()
text_columns = analysis_df.select_dtypes(include=["str"]).columns
for column in text_columns:
    analysis_df[column] = analysis_df[column].map(
        lambda value: None if isinstance(value, str) and not value.strip() else value
    )

date_columns = [
    "CREATION_DATE",
    "LAST_UPDATE",
    "LATEST_CONSTRUCTION_DATE",
    "LATEST_MAINTENANCE_DATE",
    "MORATORIUM_EXPIRATION_DATE",
    "ORIGINAL_CONSTRUCTION_DATE",
    "PCI_DATE",
]
for column in date_columns:
    analysis_df[column] = pd.to_datetime(
        analysis_df[column], unit="ms", utc=True, errors="coerce"
    ).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    analysis_df[column] = analysis_df[column].where(analysis_df[column].notna(), None)

analysis_df["GEOMETRY_REPAIRED"] = ~geometry_df["valid_source_geometry"] & ~geometry_df["empty_geometry_flag"]
analysis_df["GEOMETRY_EMPTY_QA_FLAG"] = geometry_df["empty_geometry_flag"]
analysis_df["GEOMETRY_QA_FLAG"] = analysis_df["GEOMETRY_REPAIRED"]
analysis_df["OBJECTID_DUPLICATE_QA_FLAG"] = analysis_df["OBJECTID"].duplicated(keep=False)
analysis_df["GLOBALID_DUPLICATE_QA_FLAG"] = analysis_df["GlobalID"].duplicated(keep=False)
analysis_df["NEGATIVE_LENGTH_QA_FLAG"] = df["Shape__Length"] < 0

completely_missing_fields = [column for column in df.columns if df[column].isna().all()]
analysis_df_compact = analysis_df.drop(columns=completely_missing_fields)
analysis_geometry_by_id = {}
for index, row in geometry_df.iterrows():
    geometry = row["geometry"]
    analysis_geometry_by_id[df.iloc[index]["OBJECTID"]] = (
        make_valid(geometry) if geometry is not None and not geometry.is_valid else geometry
    )

print("Completely missing source fields:", completely_missing_fields)
print("Derived analytical fields:", len(analysis_df_compact.columns))
print("QA flag totals:")
display(analysis_df.filter(like="_QA_FLAG").sum().to_frame("flagged_records"))

Completely missing source fields: ['MORATORIUM_EXPIRATION_DATE', 'ASBUILT_REFERENCE', 'CORRIDOR', 'DESCRIPTION1', 'GIS_COMMENTS', 'MORATORIUM_STATUS', 'PW_ROADWAY_ID', 'STORM_STATUS']
Derived analytical fields: 39
QA flag totals:


,flagged_records
GEOMETRY_EMPTY_QA_FLAG,0
GEOMETRY_QA_FLAG,0
OBJECTID_DUPLICATE_QA_FLAG,0
GLOBALID_DUPLICATE_QA_FLAG,0
NEGATIVE_LENGTH_QA_FLAG,0


## Validation and Export

Rows and source geometries are preserved one-for-one. Invalid polylines are repaired only in the processed derived view; valid source geometries are copied directly to avoid unnecessary coordinate rewriting.

In [5]:
assert len(raw_data["features"]) == len(analysis_df) == 66
assert analysis_df["OBJECTID"].is_unique
assert analysis_df["GlobalID"].is_unique
assert not analysis_df["GEOMETRY_EMPTY_QA_FLAG"].any()
assert all(
    geometry is not None and geometry.is_valid and geometry.length >= 0
    for geometry in analysis_geometry_by_id.values()
)
assert analysis_df_compact.columns.is_unique

print("Cleaning validation passed")
print(f"Rows preserved: {len(analysis_df):,}")
print(f"Compact fields: {len(analysis_df_compact.columns):,}")
print(f"Repaired geometries: {int(analysis_df['GEOMETRY_REPAIRED'].sum())}")
print(f"Negative source lengths flagged: {int(analysis_df['NEGATIVE_LENGTH_QA_FLAG'].sum())}")

Cleaning validation passed
Rows preserved: 66
Compact fields: 39
Repaired geometries: 0
Negative source lengths flagged: 0


In [9]:
import math


def json_safe(value):
    if value is None or value is pd.NA:
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def geometry_to_arcgis_paths(geometry):
    if geometry is None:
        return None
    lines = []
    if geometry.geom_type == "LineString":
        lines = [geometry]
    elif geometry.geom_type == "MultiLineString":
        lines = list(geometry.geoms)
    elif geometry.geom_type == "GeometryCollection":
        for item in geometry.geoms:
            if item.geom_type == "LineString":
                lines.append(item)
            elif item.geom_type == "MultiLineString":
                lines.extend(item.geoms)
    paths = [[[json_safe(x), json_safe(y)] for x, y in line.coords] for line in lines]
    return {"paths": paths} if paths else None

schema_records = [field.copy() for field in raw_data["fields"]]
source_field_names = {field["name"] for field in schema_records}
for field_name in analysis_df_compact.columns:
    if field_name not in source_field_names:
        schema_records.append({
            "name": field_name,
            "alias": field_name,
            "type": "esriFieldTypeSmallInteger",
        })

processed_features = []
for index, raw_feature in enumerate(raw_data["features"]):
    attributes = {
        field: json_safe(value)
        for field, value in analysis_df_compact.iloc[index].to_dict().items()
    }
    object_id = df.iloc[index]["OBJECTID"]
    geometry = (
        geometry_to_arcgis_paths(analysis_geometry_by_id[object_id])
        if analysis_df.iloc[index]["GEOMETRY_REPAIRED"]
        else raw_feature.get("geometry")
    )
    processed_features.append({"attributes": attributes, "geometry": geometry})

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
processed_data = {
    "objectIdFieldName": raw_data.get("objectIdFieldName"),
    "globalIdFieldName": raw_data.get("globalIdFieldName"),
    "geometryType": raw_data.get("geometryType"),
    "spatialReference": raw_data.get("spatialReference"),
    "fields": schema_records,
    "features": processed_features,
    "cleaning_summary": {
        "source_file": RAW_FILE.name,
        "data_version": DATA_VERSION,
        "rows": len(processed_features),
        "compact_attribute_fields": len(analysis_df_compact.columns),
        "completely_missing_source_fields": completely_missing_fields,
        "repaired_geometries": int(analysis_df["GEOMETRY_REPAIRED"].sum()),
        "negative_source_lengths": int(analysis_df["NEGATIVE_LENGTH_QA_FLAG"].sum()),
    },
}

with open(PROCESSED_FILE, "w", encoding="utf-8") as file:
    json.dump(processed_data, file, indent=2, ensure_ascii=True)

In [10]:
from datetime import datetime

cleaning_manifest = [
    {
        "field_or_scope": "Blank text values",
        "action": "Represent blank or whitespace-only strings as missing in derived attributes",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "CREATION_DATE, LAST_UPDATE, LATEST_CONSTRUCTION_DATE, LATEST_MAINTENANCE_DATE, MORATORIUM_EXPIRATION_DATE, ORIGINAL_CONSTRUCTION_DATE, PCI_DATE",
        "action": "Convert ArcGIS epoch milliseconds to ISO-8601 UTC strings",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "Completely missing source fields",
        "action": "Drop empty columns from the compact derived view; preserved in the raw snapshot",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "Invalid polyline geometry",
        "action": "Use make_valid in derived geometry view",
        "source_preserved": True,
        "review_flag": "GEOMETRY_QA_FLAG",
    },
    {
        "field_or_scope": "Duplicate identifiers and negative source lengths",
        "action": "Retain source records and flag for review",
        "source_preserved": True,
        "review_flag": "OBJECTID_DUPLICATE_QA_FLAG / GLOBALID_DUPLICATE_QA_FLAG / NEGATIVE_LENGTH_QA_FLAG",
    },
]

manifest_data = {
    "dataset": DATASET_ID,
    "dataset_name": source_manifest.get("dataset_name"),
    "data_version": DATA_VERSION,
    "created_utc": datetime.now().astimezone().isoformat(),
    "source_file": str(RAW_FILE),
    "processed_file": str(PROCESSED_FILE),
    "rows": len(processed_features),
    "cleaning_manifest": cleaning_manifest,
}

with open(CLEANING_MANIFEST_FILE, "w", encoding="utf-8") as file:
    json.dump(manifest_data, file, indent=2, ensure_ascii=True)

with open(PROCESSED_FILE, "r", encoding="utf-8") as file:
    reloaded_processed_data = json.load(file)
with open(CLEANING_MANIFEST_FILE, "r", encoding="utf-8") as file:
    reloaded_manifest_data = json.load(file)

assert len(reloaded_processed_data["features"]) == len(raw_data["features"])
assert len(reloaded_manifest_data["cleaning_manifest"]) == len(cleaning_manifest)
assert reloaded_manifest_data["dataset"] == DATASET_ID
assert reloaded_processed_data["features"][0]["attributes"]["OBJECTID"] == int(df.iloc[0]["OBJECTID"])
print(f"Wrote: {PROCESSED_FILE}")
print(f"Wrote: {CLEANING_MANIFEST_FILE}")
print("Reload validation passed")

Wrote: d:\Github Desktop\hillsborogis\notebooks\2026-09-08\datasets\processed\HIL-007_cleaned.json
Wrote: d:\Github Desktop\hillsborogis\notebooks\2026-09-08\manifests\HIL-007_cleaning_manifest.json
Reload validation passed
